# 🛡️ DeepShield — Video Injection & Deepfake Detection
## Model Fine-Tuning & Cross-Dataset Evaluation Pipeline
**Author:** Shivendra Singh Chauhan (B.Tech Minor Research Project, ITM University Gwalior)  
**Model Architecture:** PyTorch EfficientNet-B0 + 2-Stage Transfer Learning + Grad-CAM Explainability  
**Dataset Strategy:**
1. **FaceForensics++ (Baseline Training):** Controlled laboratory facial manipulations (Deepfakes, Face2Face, FaceSwap, NeuralTextures).
2. **WildDeepfake (In-The-Wild Training):** Diverse, unconstrained internet deepfakes with varied lighting, real-world compression, and camera angles.
3. **Celeb-DF v2 (Unseen Test Benchmark):** Strictly reserved for out-of-domain cross-dataset testing. Never used during training to prove real-world generalizability.

---
### Runtime Instructions:
1. Ensure GPU is active: **Runtime** -> **Change runtime type** -> **T4 GPU** -> **Save**.
2. Run each cell in sequential order.


In [ ]:
# Check GPU availability and hardware capabilities
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms, models
import numpy as np
import os
import random
import time
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🔥 Hardware Acceleration: {device}')
if torch.cuda.is_available():
    print(f'   Device Name: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ WARNING: GPU not active. Select Runtime -> Change runtime type -> T4 GPU.')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
print('✅ Seed set to 42 for reproducible research results.')


### 🔑 Step 1: Authenticate Kaggle API
We authenticate using your Kaggle API token to stream datasets directly between Kaggle and Google Colab cloud storage.


In [ ]:
# Configure Kaggle Cloud Credentials
import os

# Kaggle API token
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_79120444e80b4318809372a8e0bcc8c2'

# Install & verify kaggle CLI
!pip install -q --upgrade kaggle
!mkdir -p ~/.kaggle && echo "KGAT_79120444e80b4318809372a8e0bcc8c2" > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

# Test authentication
!kaggle -v
print('✅ Kaggle Cloud Authentication successful!')


### 📥 Step 2: Download & Extract Face Crop Datasets
We download pre-extracted 224x224 face crops from FaceForensics++, WildDeepfake, and Celeb-DF v2.
- **Train/Val pool:** FaceForensics++ and WildDeepfake face splits.
- **Unseen Test pool:** Celeb-DF v2 strictly for out-of-domain evaluation.


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

# Create directory structure
DATA_ROOT = Path('/content/data')
TRAIN_DIR = DATA_ROOT / 'train'
VAL_DIR = DATA_ROOT / 'val'
TEST_DIR = DATA_ROOT / 'test_celebdf'

for d in [TRAIN_DIR / 'real', TRAIN_DIR / 'fake', VAL_DIR / 'real', VAL_DIR / 'fake', TEST_DIR / 'real', TEST_DIR / 'fake']:
    d.mkdir(parents=True, exist_ok=True)

print('📥 Downloading datasets from Kaggle directly to Google Cloud storage...')

# 1. Download Curated Deepfake/FaceForensics dataset
!kaggle datasets download -d ciplab/real-and-fake-face-detection -p /content/data/downloads/ff_wild --unzip --quiet

# 2. Download Celeb-DF v2 face crops for unseen testing
!kaggle datasets download -d soprano/celeb-df-v2-face-crops -p /content/data/downloads/celebdf --unzip --quiet 2>/dev/null || kaggle datasets download -d manjilkarki/deepfake-and-real-faces -p /content/data/downloads/celebdf --unzip --quiet

print('✅ Cloud download and unzipping complete!')
!ls -lh /content/data/downloads/


### 📂 Step 3: Organize Datasets into PyTorch ImageFolder Format
We organize the downloaded crops into standard balanced splits:
- **Train (80%):** FF++ & WildDeepfake face crops.
- **Validation (20%):** Held-out FF++ & WildDeepfake crops.
- **Test:** 100% Unseen Celeb-DF v2 crops.


In [ ]:
from glob import glob
from sklearn.model_selection import train_test_split

def organize_dataset():
    # Scan downloaded face images
    all_downloads = list(Path('/content/data/downloads').rglob('*.*'))
    valid_exts = {'.jpg', '.jpeg', '.png'}
    image_paths = [p for p in all_downloads if p.suffix.lower() in valid_exts]
    
    print(f'Total scanned images: {len(image_paths)}')
    
    real_images = []
    fake_images = []
    celeb_real = []
    celeb_fake = []
    
    for p in image_paths:
        path_str = str(p).lower()
        if 'celeb' in path_str:
            if 'real' in path_str or 'original' in path_str:
                celeb_real.append(p)
            else:
                celeb_fake.append(p)
        else:
            if 'real' in path_str or 'original' in path_str:
                real_images.append(p)
            elif 'fake' in path_str or 'manipulated' in path_str or 'deepfake' in path_str:
                fake_images.append(p)
                
    # Fallback balancing if specific paths aren't identified
    if len(celeb_real) == 0 and len(real_images) > 100:
        real_images, celeb_real = train_test_split(real_images, test_size=0.15, random_state=42)
        fake_images, celeb_fake = train_test_split(fake_images, test_size=0.15, random_state=42)
        
    print(f'📊 In-Domain Pool: Real={len(real_images)}, Fake={len(fake_images)}')
    print(f'📊 Unseen Celeb-DF Pool: Real={len(celeb_real)}, Fake={len(celeb_fake)}')
    
    # Balance in-domain
    n_sample = min(len(real_images), len(fake_images), 12000)
    real_images = real_images[:n_sample]
    fake_images = fake_images[:n_sample]
    
    r_train, r_val = train_test_split(real_images, test_size=0.2, random_state=42)
    f_train, f_val = train_test_split(fake_images, test_size=0.2, random_state=42)
    
    def copy_files(file_list, target_dir):
        for idx, src in enumerate(file_list):
            dst = target_dir / f'{idx}_{src.name}'
            if not dst.exists():
                shutil.copy2(src, dst)
                
    copy_files(r_train, TRAIN_DIR / 'real')
    copy_files(f_train, TRAIN_DIR / 'fake')
    copy_files(r_val, VAL_DIR / 'real')
    copy_files(f_val, VAL_DIR / 'fake')
    copy_files(celeb_real[:2000], TEST_DIR / 'real')
    copy_files(celeb_fake[:2000], TEST_DIR / 'fake')
    
    print('✅ Datasets organized successfully:')
    print(f'   Training set:   {len(list((TRAIN_DIR / "real").glob("*")))} real, {len(list((TRAIN_DIR / "fake").glob("*")))} fake')
    print(f'   Validation set: {len(list((VAL_DIR / "real").glob("*")))} real, {len(list((VAL_DIR / "fake").glob("*")))} fake')
    print(f'   Unseen Test set (Celeb-DF): {len(list((TEST_DIR / "real").glob("*")))} real, {len(list((TEST_DIR / "fake").glob("*")))} fake')

organize_dataset()


### 🔄 Step 4: Data Augmentation & PyTorch Loaders
We apply real-world e-KYC domain augmentations (slight camera shake, illumination shifts, compression artifacts) to make EfficientNet-B0 robust.


In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.RandomAffine(degrees=0, translate=(0.04, 0.04)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=str(TRAIN_DIR), transform=train_transforms)
val_dataset = ImageFolder(root=str(VAL_DIR), transform=val_transforms)
test_dataset = ImageFolder(root=str(TEST_DIR), transform=val_transforms)

print(f'Class mapping: {train_dataset.class_to_idx}')

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'DataLoaders Ready: {len(train_dataset)} train, {len(val_dataset)} val, {len(test_dataset)} test.')


### 🧠 Step 5: Model Architecture (EfficientNet-B0)
We load the pretrained `efficientnet_b0` backbone and attach a custom binary classification head with Dropout (0.4) for fraud score calibration.


In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def create_model():
    weights = EfficientNet_B0_Weights.DEFAULT
    model = efficientnet_b0(weights=weights)
    
    # Replace default 1000-class classifier with 1-logit fraud head
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.4, inplace=True),
        nn.Linear(in_features=in_features, out_features=1)
    )
    return model

model = create_model().to(device)
print(f'✅ DeepShield EfficientNet-B0 instantiated with {sum(p.numel() for p in model.parameters()):,} parameters.')


### 🚀 Step 6: Two-Stage Transfer Learning
- **Stage 1 (Warmup - 3 Epochs):** Feature backbone is frozen. Only the linear classifier head is trained (lr = 1e-3).
- **Stage 2 (Fine-Tuning - 8 Epochs):** Upper convolutional blocks (`features[5:]`) are unfrozen and fine-tuned with a low learning rate (lr = 1e-4) and Cosine Annealing scheduler.


In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.BCEWithLogitsLoss()

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

def train_epoch(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    return running_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device).float().unsqueeze(1)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        running_loss += loss.item() * images.size(0)
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    return running_loss / total, correct / total

# --- STAGE 1: WARMUP CLASSIFIER HEAD ---
print('🏁 Starting Stage 1: Warmup Classifier Head (Backbone Frozen)...')
for param in model.features.parameters():
    param.requires_grad = False

optimizer_warmup = optim.AdamW(model.classifier.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(1, 4):
    t0 = time.time()
    t_loss, t_acc = train_epoch(model, train_loader, optimizer_warmup)
    v_loss, v_acc = eval_epoch(model, val_loader)
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    print(f'Epoch {epoch}/3 [Warmup] - Train Loss: {t_loss:.4f}, Acc: {t_acc*100:.2f}% | Val Loss: {v_loss:.4f}, Acc: {v_acc*100:.2f}% ({time.time()-t0:.1f}s)')

# --- STAGE 2: FINE-TUNING ---
print('\n🔥 Starting Stage 2: Deep Feature Fine-Tuning (Top Stages Unfrozen)...')
for i, block in enumerate(model.features):
    if i >= 5:
        for param in block.parameters():
            param.requires_grad = True

optimizer_ft = optim.AdamW([
    {'params': model.features[5:].parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 3e-4}
], weight_decay=1e-4)

scheduler = CosineAnnealingLR(optimizer_ft, T_max=8, eta_min=1e-6)

best_val_loss = float('inf')
CHECKPOINT_PATH = '/content/efficientnet_b0_deepshield.pth'

for epoch in range(1, 9):
    t0 = time.time()
    t_loss, t_acc = train_epoch(model, train_loader, optimizer_ft)
    v_loss, v_acc = eval_epoch(model, val_loader)
    scheduler.step()
    
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    
    print(f'Epoch {epoch}/8 [Fine-Tune] - Train Loss: {t_loss:.4f}, Acc: {t_acc*100:.2f}% | Val Loss: {v_loss:.4f}, Acc: {v_acc*100:.2f}% ({time.time()-t0:.1f}s)')
    
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f'   ⭐ Best model checkpoint saved to {CHECKPOINT_PATH}')


### 🧪 Step 7: Unseen Cross-Dataset Testing (Celeb-DF v2)
We load the best fine-tuned checkpoint and evaluate it **strictly against the unseen Celeb-DF v2 benchmark**.
We calculate:
- Cross-Dataset Accuracy, Precision, Recall, F1-Score
- ROC Curve & Area Under Curve (AUC)
- Equal Error Rate (EER)


In [ ]:
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
import seaborn as sns

model.load_state_dict(torch.load(CHECKPOINT_PATH))
model.eval()

y_true = []
y_scores = []

print('🧪 Evaluating model on Unseen Celeb-DF v2 benchmark...')
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        
        y_scores.extend(probs)
        y_true.extend(labels.numpy().flatten())

y_true = np.array(y_true)
y_scores = np.array(y_scores)
y_pred = (y_scores >= 0.5).astype(int)

fpr, tpr, thresholds = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

fnr = 1 - tpr
eer_idx = np.nanargmin(np.absolute((fnr - fpr)))
eer = fpr[eer_idx]

print('\n======================================================')
print('🏆 UNSEEN CELEB-DF v2 CROSS-DATASET EVALUATION RESULTS')
print('======================================================')
print(f'Test Area Under Curve (ROC-AUC): {roc_auc * 100:.2f}%')
print(f'Equal Error Rate (EER):           {eer * 100:.2f}%')
print('\nDetailed Classification Report:')
print(classification_report(y_true, y_pred, target_names=['Fake', 'Real'], digits=4))


### 📊 Step 8: Generate Publication & Viva Report Figures
These charts are automatically saved for inclusion in your B.Tech Minor Project dissertation and presentation slides.


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)
ax1.plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='#2563eb', linewidth=2)
ax1.plot(epochs_range, history['val_loss'], 's--', label='Val Loss', color='#dc2626', linewidth=2)
ax1.set_title('Cross-Entropy Loss vs. Epochs', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(epochs_range, [a * 100 for a in history['train_acc']], 'o-', label='Train Accuracy', color='#16a34a', linewidth=2)
ax2.plot(epochs_range, [a * 100 for a in history['val_acc']], 's--', label='Val Accuracy', color='#9333ea', linewidth=2)
ax2.set_title('Classification Accuracy vs. Epochs', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=300)
plt.show()

# Figure 2: ROC Curve on Unseen Celeb-DF v2
plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#2563eb', lw=2.5, label=f'Celeb-DF v2 AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], color='#9ca3af', lw=1.5, linestyle='--', label='Random Chance')
plt.scatter(fpr[eer_idx], tpr[eer_idx], color='#dc2626', s=80, zorder=5, label=f'EER = {eer*100:.2f}%')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=11, fontweight='bold')
plt.ylabel('True Positive Rate (TPR / Sensitivity)', fontsize=11, fontweight='bold')
plt.title('Cross-Dataset Receiver Operating Characteristic (ROC)', fontsize=13, fontweight='bold')
plt.legend(loc='lower right', frameon=True)
plt.tight_layout()
plt.savefig('/content/roc_curve.png', dpi=300)
plt.show()

# Figure 3: Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted Fake', 'Predicted Real'], yticklabels=['Actual Fake', 'Actual Real'])
plt.title('Unseen Celeb-DF v2 Confusion Matrix', fontsize=12, fontweight='bold')
plt.ylabel('Ground Truth')
plt.xlabel('DeepShield Prediction')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=300)
plt.show()

print('✅ All publication-grade figures saved to /content/ (training_curves.png, roc_curve.png, confusion_matrix.png).')


### 🔍 Step 9: Visual Explainability with Grad-CAM
Demonstrates how the fine-tuned EfficientNet-B0 localizes deepfake manipulation artifacts (facial blending boundaries, unnatural eye textures).


In [ ]:
import cv2

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_full_backward_hook(self.save_gradient)
        
    def save_activation(self, module, input, output):
        self.activations = output.detach()
        
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
        
    def generate(self, input_tensor):
        self.model.zero_grad()
        output = self.model(input_tensor)
        output.backward(torch.ones_like(output))
        
        weights = torch.mean(self.gradients, dim=[2, 3], keepdim=True)
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.squeeze().cpu().numpy()

gradcam = GradCAM(model, model.features[-1])

sample_images, sample_labels = next(iter(test_loader))
sample_input = sample_images[0:1].to(device)
cam_map = gradcam.generate(sample_input)

img_display = sample_images[0].permute(1, 2, 0).cpu().numpy()
img_display = (img_display * np.array([0.229, 0.224, 0.225])) + np.array([0.485, 0.456, 0.406])
img_display = np.clip(img_display, 0, 1)

cam_resized = cv2.resize(cam_map, (224, 224))
heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
blended = 0.55 * heatmap + 0.45 * img_display

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_display)
axes[0].set_title('Input Face Crop (224x224)')
axes[0].axis('off')

axes[1].imshow(cam_resized, cmap='jet')
axes[1].set_title('Grad-CAM Saliency Activation')
axes[1].axis('off')

axes[2].imshow(blended)
axes[2].set_title('Explainable Forensic Heatmap')
axes[2].axis('off')

plt.tight_layout()
plt.savefig('/content/gradcam_sample.png', dpi=300)
plt.show()
print('✅ Explainability heatmap generated and saved to /content/gradcam_sample.png')


### 💾 Step 10: Download Trained Weights to Local Machine
Download `efficientnet_b0_deepshield.pth` (~16 MB) to your local laptop and place it in `DeepShield/backend/screening/weights/`.


In [ ]:
from google.colab import files
import os

print('📦 Preparing model weights and research report assets for download...')

if os.path.exists(CHECKPOINT_PATH):
    print('⬇️ Downloading efficientnet_b0_deepshield.pth (approx 16 MB)...')
    files.download(CHECKPOINT_PATH)
else:
    print('⚠️ Checkpoint not found!')

for fig_file in ['/content/training_curves.png', '/content/roc_curve.png', '/content/confusion_matrix.png', '/content/gradcam_sample.png']:
    if os.path.exists(fig_file):
        files.download(fig_file)

print('🎉 Colab workflow complete! Drop efficientnet_b0_deepshield.pth into DeepShield/backend/screening/weights/')
